In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load CSVs and extract the 'MSE' column
def load_mse_from_csv(filepath, mse_column='MSE'):
    df = pd.read_csv(filepath)
    if mse_column not in df.columns:
        raise ValueError(f"Column '{mse_column}' not found in {filepath}")
    return df[mse_column].values

# Load MSE data from multi-column CSVs
clean_all = load_mse_from_csv('/content/baseline.csv', mse_column='MSE')
fraud_all = load_mse_from_csv('/content/fcs_data_6.csv', mse_column='MSE')

# Sample 1,000,000 from each group
np.random.seed(42)
clean_sample = np.random.choice(clean_all, size=1_000_000, replace=False)
fraud_sample = np.random.choice(fraud_all, size=1_000_000, replace=False)

def plot_all(clean_data, fraud_data, title_prefix, bins):
    # Plot 1: Density
    plt.figure(figsize=(8,6))
    sns.histplot(clean_data, bins=bins, stat='density', kde=False, color='green', label='Baseline', alpha=0.5)
    sns.histplot(fraud_data, bins=bins, stat='density', kde=False, color='yellow', label='Clean', alpha=0.5)
    plt.title(f"{title_prefix} - Density Distribution")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Overlap for Plot 1
    hist_clean, _ = np.histogram(clean_data, bins=bins, density=True)
    hist_fraud, _ = np.histogram(fraud_data, bins=bins, density=True)
    min_density = np.minimum(hist_clean, hist_fraud)
    overlap_area = np.sum(min_density * np.diff(bins))
    print(f"{title_prefix} - Overlap (Density Distribution): {overlap_area * 100:.2f}%")

    # Plot 2: Count
    plt.figure(figsize=(8,6))
    sns.histplot(clean_data, bins=bins, stat='count', color='green', label='Baseline', alpha=0.5)
    sns.histplot(fraud_data, bins=bins, stat='count', color='yellow', label='Clean', alpha=0.5)
    plt.title(f"{title_prefix} - Count Distribution")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Plot 3: MSE ≥ 0.01843
    clean_filtered = clean_data[clean_data >= 0.01843]
    fraud_filtered = fraud_data[fraud_data >= 0.01843]
    bins_02 = np.linspace(0.01843, max(clean_filtered.max(), fraud_filtered.max()), 100)

    plt.figure(figsize=(8,6))
    sns.histplot(clean_filtered, bins=bins_02, stat='density', color='green', label='Baseline', alpha=0.5)
    sns.histplot(fraud_filtered, bins=bins_02, stat='density', color='yellow', label='Clean', alpha=0.5)
    plt.title(f"{title_prefix} - Density from MSE ≥ 0.01843")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Overlap for filtered MSEs
    hist_clean_02, _ = np.histogram(clean_filtered, bins=bins_02, density=True)
    hist_fraud_02, _ = np.histogram(fraud_filtered, bins=bins_02, density=True)
    min_density_02 = np.minimum(hist_clean_02, hist_fraud_02)
    overlap_area_02 = np.sum(min_density_02 * np.diff(bins_02))
    print(f"{title_prefix} - Overlap (MSE ≥ 0.01843): {overlap_area_02 * 100:.2f}%")

    # Plot 4: Fraud MSEs with threshold
    percent_unhealthy = 3.28
    threshold_index = int((percent_unhealthy / 100) * len(fraud_data))
    threshold_value = np.sort(fraud_data)[-threshold_index]

    plt.figure(figsize=(8,6))
    sns.histplot(fraud_data, bins=bins, kde=False, color='blue', label="Fraud MSEs", alpha=0.6)
    plt.axvline(threshold_value, color='red', linestyle='--', label=f"Threshold = {threshold_value:.5f}")
    plt.title(f"{title_prefix} - Fraud MSEs with {percent_unhealthy}% Threshold")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Overlap for full data
    hist_clean, _ = np.histogram(clean_data, bins=bins, density=True)
    hist_fraud, _ = np.histogram(fraud_data, bins=bins, density=True)
    min_density = np.minimum(hist_clean, hist_fraud)
    overlap_area = np.sum(min_density * np.diff(bins))
    print(f"{title_prefix} - Overlap (All MSEs): {overlap_area * 100:.2f}%\n")


# Create bin ranges
bins_sample = np.linspace(min(clean_sample.min(), fraud_sample.min()),
                          max(clean_sample.max(), fraud_sample.max()), 100)
bins_all = np.linspace(min(clean_all.min(), fraud_all.min()),
                       max(clean_all.max(), fraud_all.max()), 100)

# Plot using sampled data
plot_all(clean_sample, fraud_sample, "Sampled (1M each)", bins_sample)

# Plot using full data
plot_all(clean_all, fraud_all, "All Data", bins_all)
